# Consultas em séries industriais com Python e orientação a objetos
  **MECAI — Mestrado Profissional em Matemática, Estatística e Computação Aplicadas à Indústria • ICMC–USP**

  **Trabalho individual • prazo: 10 dias após a disponibilização pelo professor.**
  
  Nome: ____________________  |  Número USP: ____________________

  Uma ocorrência industrial exige recuperar medições de sensores em instantes específicos e em janelas de investigação. Implemente um serviço reutilizável de consultas, compare os métodos estudados e recomende estratégias com base em evidências.


## Objetivos, regras e avaliação
  Desenvolver busca binária, sequencial indexada, por salto, interpolação e exponencial; utilizar herança, encapsulamento, composição e polimorfismo; validar consultas pontuais e por intervalo; distinguir correção, comparações e tempo físico.

  | Critério | Peso |
  |---|---:|
  | Métodos pontuais e limites binários/intervalos | 40% |
  | Orientação a objetos efetiva | 25% |
  | Casos especiais, invariantes e independência | 15% |
  | Experimentos e recomendação técnica | 20% |

  **OO é obrigatória:** todas as estratégias herdam de `MetodoBusca`; o serviço recebe objetos e delega consultas pelo contrato comum. Uma solução composta apenas de funções, ou uma classe que escolhe algoritmos por nomes em uma cadeia de condicionais, não atende ao critério de OO. Não crie índices globais, não reconstrua o índice a cada busca e não use variáveis de classe para estado de sensores. Os testes verificam propriedades observáveis; o professor também revisará a estrutura do código.

  Não use `bisect`, `numpy.searchsorted`, dicionários de instante→posição, `list.index` ou funções prontas de busca nas suas estratégias. `dict` é permitido para mapear identificadores de sensores no serviço. A infraestrutura fornecida pode usar referências prontas para verificar seus resultados.

  Estimativa: 12–16 horas. Dias 1–2: modelo e revisão binária; 3–6: estratégias; 7–8: serviço e testes; 9–10: experimentos e análise. Entrega: somente este `.ipynb`, com implementações, resultados e respostas. Todo o trabalho é sequencial; concorrência fica fora do escopo.


## 1. Ambiente e situação industrial
  Abra no Colab com ambiente CPU e execute as células em ordem. Não é preciso conectar Drive nem obter dados externos. Os dados sintéticos simulam aquisição periódica, falhas de comunicação e coleta por eventos; não são medições reais nem estabelecem diagnósticos físicos.

  Cada medição tem `sensor_id`, `instante` inteiro e `valor` numérico finito. Para cada sensor, os instantes são estritamente crescentes e únicos; sensores distintos podem compartilhar instantes. A série deve ser imutável depois de construída. Não ordene silenciosamente uma entrada inválida: sinalize `ValueError`.

  As posições começam em zero. Consulta pontual devolve posição ou −1. O serviço diferencia sensor inexistente e instante ausente. A janela é fechada: `[inicio,fim]`.

  As células de implementação estão vazias. Sugestões e contratos ficam nas células de texto. A classe abstrata e os códigos de teste/experimento são fornecidos e não devem ser alterados para esconder falhas.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, FrozenInstanceError
from math import isfinite, isqrt
from time import perf_counter
from statistics import median
import bisect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print('Ambiente pronto. Python com CPU é suficiente; não há arquivos externos.')


## 2. Exercício A — Modelo de dados e invariantes
  Implemente as cinco classes abaixo. É permitido usar `@dataclass(frozen=True)`; se escolher outra solução, preserve interfaces e comportamento. A série deve converter a coleção recebida para tupla e rejeitar: instantes repetidos/fora de ordem; medição de outro sensor; identificador vazio; instante não inteiro; valor não finito.

  | Classe | Construtor e atributos públicos obrigatórios |
  |---|---|
  | `Medicao` | `Medicao(sensor_id: str, instante: int, valor: float)` |
  | `SerieSensor` | `SerieSensor(sensor_id, medicoes)`; propriedades `sensor_id`, `medicoes` (tupla), `instantes` (tupla); `len(serie)` |
  | `ResultadoBusca` | `ResultadoBusca(posicao, comparacoes)`; propriedade `encontrado` |
  | `RespostaConsulta` | `RespostaConsulta(status, medicao, resultado)` |
  | `ResumoPeriodo` | `ResumoPeriodo(medicoes, comparacoes)`; propriedades `quantidade`, `minimo`, `maximo`, `media` |

  Em `RespostaConsulta`, os status são `encontrada`, `instante_ausente`, `sensor_inexistente`. No último caso, `medicao` e `resultado` são `None`; no instante ausente, `medicao` é `None` mas há `ResultadoBusca(-1,c)`.

  Para janela vazia: quantidade zero e mínimo, máximo e média `None`. Os registros retornados devem preservar ordem. Propriedades estatísticas podem ser calculadas sob demanda; discuta seu custo. Use `FrozenInstanceError` como exceção para tentativa de atribuir campos imutáveis, para compatibilidade com os testes fornecidos.

  Sugestão de declaração parcial (implemente as demais e as validações):
  ```python
  @dataclass(frozen=True)
  class ResultadoBusca:
      posicao: int
      comparacoes: int
      # acrescentar propriedade encontrado
  ```


**Resposta A — explique duas invariantes e sua importância**

_Escreva sua resposta aqui._


## 3. Infraestrutura OO fornecida — contrato abstrato
  **Execute a célula a seguir depois de implementar o modelo.** A classe não realiza nenhuma busca; centraliza preparação, verificação de estado e contagem. Você implementará `_buscar` nas subclasses. Use `_preparar` apenas quando houver trabalho adicional e chame `super().__init__()` nos construtores próprios.

  Toda comparação entre chaves usa `_comparar(a,b)` e conta **uma comparação tricotômica**, com retorno −1/0/1. Comparações de índices não entram na métrica; o modelo não é contagem de instruções Python. Não reconte a validação de SerieSensor como parte da busca. O contador reinicia por consulta; a busca exponencial deve somar o contador de sua binária auxiliar.

  `preparar` pode associar novamente a estratégia a uma nova série; nesse caso o estado anterior deve ser substituído. O serviço não permite usar a mesma instância em sensores distintos simultaneamente. Não altere os métodos públicos `preparar` e `buscar` nas subclasses.


In [ ]:
class MetodoBusca(ABC):
    """Infraestrutura fornecida: preparação, instrumentação e contrato polimórfico.
    Uso sequencial; este contador mutável não foi projetado para concorrência.
    """
    def __init__(self):
        self._serie=None
        self._comparacoes=0
    def preparar(self, serie):
        if not isinstance(serie,SerieSensor):
            raise TypeError('Esperada uma SerieSensor')
        self._serie=serie
        self._preparar()
    def _preparar(self):
        pass
    @property
    def serie(self):
        if self._serie is None:
            raise RuntimeError('Chame preparar antes de consultar')
        return self._serie
    def _comparar(self,a,b):
        self._comparacoes+=1
        return (a>b)-(a<b)
    def buscar(self,instante):
        _=self.serie
        self._comparacoes=0
        posicao=self._buscar(instante)
        return ResultadoBusca(posicao,self._comparacoes)
    @abstractmethod
    def _buscar(self,instante):
        """Devolver posição ou -1. Usar _comparar para comparar chaves."""
        raise NotImplementedError
    @property
    def entradas_indice(self):
        return 0


## 4. Exercício B — Busca binária e limites de uma janela
  Implemente `BuscaBinaria(MetodoBusca)`. Métodos obrigatórios:
  - `_buscar(instante) -> int`;
  - `buscar_intervalo(instante, lo, hi) -> ResultadoBusca`, usando intervalo **fechado** e reiniciando o contador;
  - `limite_inferior(instante) -> (posicao, comparacoes)`: primeira posição com chave ≥ instante;
  - `limite_superior(instante) -> (posicao, comparacoes)`: primeira posição com chave > instante.

  Os limites retornam `n` se não há posição elegível, inclusive zero para série vazia. Não usam −1. Sugestão: uma função privada compartilhada para os dois limites; outra para a busca exata, reaproveitada por `_buscar` e `buscar_intervalo`.

  ```text
  BUSCA_EXATA(x, lo, hi):
      enquanto lo ≤ hi:
          meio ← lo + piso((hi−lo)/2)
          r ← comparar(A[meio], x)
          se r = 0: devolver meio
          se r < 0: lo ← meio+1; senão: hi ← meio−1
      devolver −1

  LIMITE(x, superior):
      contador ← 0; lo ← 0; hi ← n   [intervalo de posições semiaberto]
      enquanto lo < hi:
          meio ← piso((lo+hi)/2)
          r ← comparar(A[meio], x)
          se r < 0 ou (superior e r = 0): lo ← meio+1
          senão: hi ← meio
      devolver (lo, contador)
  ```

  Questões: B1. Por que `hi=n` não causa acesso a `A[n]` nos limites? B2. Para `[10,20,30]`, quais limites delimitam `[15,30]`? B3. Por que os limites não são iguais à busca exata?


**Respostas B**

_Escreva sua resposta aqui._


## 5. Exercício C — Busca sequencial indexada
  Crie `BuscaIndexada(bloco)`. Rejeite bloco não inteiro ou ≤0. Na preparação, construa um índice privado com entradas `(maximo, inicio, fim_exclusivo)`. A propriedade `entradas_indice` deve retornar sua quantidade.

  ```text
  PREPARAR:
      índice ← vazio
      para início em 0, b, 2b, ... enquanto início < n:
          fim ← mínimo(início+b, n)
          acrescentar (A[fim−1], início, fim) ao índice
  BUSCAR(x):
      para máximo, início, fim no índice:
          se comparar(máximo, x) ≥ 0:
              para i de início até fim−1:
                  r ← comparar(A[i], x)
                  se r = 0: devolver i
                  se r > 0: devolver −1
              devolver −1
      devolver −1
  ```

  Não reconstrua o índice em `_buscar`. Para o vetor de instantes já disponível, a preparação apresentada preenche `ceil(n/b)` entradas, sem varrer todos os registros. Explique seu tempo, memória e custo de consulta. Ao preparar série vazia ou diferente, elimine entradas anteriores.


**Resposta C**

_Escreva sua resposta aqui._


## 6. Exercício D — Busca por salto
  Crie `BuscaSalto(bloco)`, com a mesma validação de bloco. Não armazene um índice.
  ```text
  BUSCAR(x):
      para início em 0, b, 2b, ... enquanto início < n:
          fim ← mínimo(início+b−1, n−1)
          se comparar(A[fim], x) ≥ 0:
              para i de início até fim:
                  r ← comparar(A[i], x)
                  se r = 0: devolver i
                  se r > 0: devolver −1
              devolver −1
      devolver −1
  ```
  Compare seu espaço auxiliar e sua contagem de comparações com a indexada, para o mesmo bloco e consulta. Igualdade de contagem implica igualdade de tempo? Justifique.


**Resposta D**

_Escreva sua resposta aqui._


## 7. Exercício E — Interpolação
  Crie `BuscaInterpolacao`. Como as chaves são inteiros Python, calcule a posição com aritmética inteira, sem converter instantes para float:
  ```text
  lo ← 0; hi ← n−1
  enquanto lo ≤ hi:
      se comparar(x,A[lo]) < 0: devolver −1
      se comparar(x,A[hi]) > 0: devolver −1
      se comparar(A[lo],A[hi]) = 0:
          devolver lo se comparar(x,A[lo]) = 0; senão −1
      pos ← lo + ((x−A[lo])*(hi−lo)) DIV (A[hi]−A[lo])
      limitar pos a [lo,hi]
      r ← comparar(A[pos],x)
      se r = 0: devolver pos
      se r < 0: lo ← pos+1; senão: hi ← pos−1
  devolver −1
  ```
  `DIV` corresponde a `//`; os operandos do quociente são não negativos após testar as extremidades. Python int não tem overflow de largura fixa, mas operações com inteiros enormes não têm custo constante.

  Questões: E1. Por que proteger o denominador, mesmo com instantes únicos? E2. Por que o intervalo diminui? E3. Diferencie pior caso O(n), custo esperado clássico sob distribuição uniforme e resultados de uma sequência exatamente periódica.


**Respostas E**

_Escreva sua resposta aqui._


## 8. Exercício F — Exponencial por composição
  Crie `BuscaExponencial` com uma instância privada de `BuscaBinaria`. Na preparação, associe a auxiliar à mesma série; na busca, reutilize `buscar_intervalo` e some suas comparações.
  ```text
  se n = 0: devolver −1
  r ← comparar(A[0],x)
  se r = 0: devolver 0
  se r > 0: devolver −1
  k ← 1
  enquanto k < n e comparar(A[k],x) < 0: k ← 2*k
  resultado ← binária_auxiliar.buscar_intervalo(x, piso(k/2), mínimo(k,n−1))
  acrescentar resultado.comparações ao contador da exponencial
  devolver resultado.posição
  ```

  A condição de limite deve ser avaliada antes do acesso. Explique o custo O(log(i+1)) para posição i, a posição de inserção na ausência e o trabalho adicional para consultas espalhadas. Composição é obrigatória aqui: não copie o laço binário para esta classe.


**Resposta F**

_Escreva sua resposta aqui._


## 9. Exercício G — Serviço polimórfico e investigação
  Implemente `ServicoConsultas` com:
  ```python
  registrar_sensor(self, serie: SerieSensor, metodo: MetodoBusca) -> None
  consultar(self, sensor_id: str, instante: int) -> RespostaConsulta
  consultar_lote(self, solicitacoes) -> list
  investigar_periodo(self, sensor_id: str, inicio: int, fim: int) -> ResumoPeriodo
  ```
  - `registrar_sensor`: preparar a estratégia uma vez, guardar série e método em um dicionário por sensor, e preparar uma `BuscaBinaria` auxiliar para limites. Registrar um sensor já existente substitui sua série e estratégia. Rejeitar com `ValueError` o compartilhamento da mesma instância em sensores distintos.
  - `consultar`: localizar o sensor no dicionário, delegar a `metodo.buscar` e encapsular o resultado. **Não selecionar algoritmos por nome/tipo.**
  - `consultar_lote`: lista de pares `(sensor_id, instante)`; preservar ordem e devolver uma resposta para cada solicitação.
  - `investigar_periodo`: usar limite inferior do início e limite superior do fim; recuperar a fatia de medições; somar as comparações dos limites. Sensor inexistente gera `KeyError`; início>fim gera `ValueError`; janela sem dados gera resumo vazio.

  A escolha de limites binários é um componente separado da estratégia de consulta pontual. Para recuperar k registros, o custo é O(log(n+1)+k). Não use varredura de toda a série para selecionar a janela.


**Resposta G — arquitetura e custo**

_Escreva sua resposta aqui._


## 10. Referências de resultado e testes fornecidos
  Os testes seguintes verificam invariantes, consultas, contadores reiniciados, preparação repetida, limites, intervalos, substituição de estratégias e sensores independentes. Um objeto espião confirma a delegação sem impor o nome de um algoritmo. Estes testes não provam o estilo OO nem a complexidade: esses aspectos também serão revisados no código.

  Caso uma célula vazia ainda não tenha sido preenchida, a infraestrutura informará as classes que faltam. Falhas de teste devem ser corrigidas, não removidas. Uma implementação com laço sem progresso pode exigir interromper a execução no Colab.


In [ ]:
REQUERIDAS=['Medicao','SerieSensor','ResultadoBusca','RespostaConsulta','ResumoPeriodo',
            'BuscaBinaria','BuscaIndexada','BuscaSalto','BuscaInterpolacao','BuscaExponencial','ServicoConsultas']
def implementado():
    faltam=[nome for nome in REQUERIDAS if nome not in globals()]
    if faltam:
        print('Implemente e execute as células das classes:',', '.join(faltam))
        return False
    return True

def serie_de(instantes,sensor_id='S1'):
    return SerieSensor(sensor_id,[Medicao(sensor_id,int(t),float(i)) for i,t in enumerate(instantes)])

def fabricas(bloco):
    return {'binaria':lambda:BuscaBinaria(), 'indexada':lambda:BuscaIndexada(bloco),
            'salto':lambda:BuscaSalto(bloco), 'interpolacao':lambda:BuscaInterpolacao(),
            'exponencial':lambda:BuscaExponencial()}

def referencia_pontual(serie,instante):
    p=bisect.bisect_left(serie.instantes,instante)
    return p if p<len(serie) and serie.instantes[p]==instante else -1

def referencia_periodo(serie,inicio,fim):
    return tuple(m for m in serie.medicoes if inicio<=m.instante<=fim)

def esperar_erro(tipo,funcao):
    try: funcao()
    except tipo: return
    raise AssertionError(f'Era esperada uma exceção {tipo.__name__}')
print('Referências prontas: bisect e varredura são utilizados somente pela infraestrutura.')


In [ ]:
def testar():
    esperar_erro(TypeError,lambda:MetodoBusca())
    esperar_erro(ValueError,lambda:serie_de([2,1]))
    esperar_erro(ValueError,lambda:serie_de([1,1]))
    esperar_erro(ValueError,lambda:SerieSensor('S2',[Medicao('S1',1,1.0)]))
    s=serie_de([2,4,6])
    esperar_erro(FrozenInstanceError,lambda:setattr(s,'medicoes',()))
    esperar_erro(FrozenInstanceError,lambda:setattr(s.medicoes[0],'valor',99.0))
    for classe in [BuscaIndexada,BuscaSalto]:
        esperar_erro(ValueError,lambda c=classe:c(0))
        esperar_erro(ValueError,lambda c=classe:c(-3))
    rng=np.random.default_rng(503)
    casos=[[],[5],[-10,0,10],[1,2,3,10**12],list(range(0,57,3))]
    casos += [sorted(rng.choice(np.arange(-1000,1001),size=n,replace=False).tolist()) for n in [3,31,100]]
    consultas_verificadas=0
    for b in [1,3,500]:
        for nome,fabrica in fabricas(b).items():
            metodo=fabrica()
            assert isinstance(metodo,MetodoBusca)
            esperar_erro(RuntimeError,lambda:metodo.buscar(5))
            for a in casos:
                serie=serie_de(a); metodo.preparar(serie)
                qs=a+[-1001,-1,0,1,1001,10**12+1]
                for x in qs:
                    r=metodo.buscar(x)
                    assert isinstance(r,ResultadoBusca)
                    assert r.posicao==referencia_pontual(serie,x),(nome,a,x,r)
                    assert r.encontrado==(r.posicao>=0)
                    assert isinstance(r.comparacoes,int) and r.comparacoes>=0
                    assert metodo.buscar(x)==r, 'O contador deve reiniciar em cada consulta'
                    consultas_verificadas+=1
                if nome=='indexada':
                    assert metodo.entradas_indice==(len(a)+b-1)//b
                else: assert metodo.entradas_indice==0
            print(nome,'bloco',b,'— aprovado')
    binary=BuscaBinaria(); binary.preparar(s)
    assert binary.buscar(4).comparacoes==1
    for x in [-1,2,3,6,9]:
        assert binary.limite_inferior(x)[0]==bisect.bisect_left(s.instantes,x)
        assert binary.limite_superior(x)[0]==bisect.bisect_right(s.instantes,x)
    for a in casos:
        serie=serie_de(a); binary.preparar(serie)
        for x in [-1001,0,1,1001,10**12+1]:
            assert binary.limite_inferior(x)[0]==bisect.bisect_left(serie.instantes,x)
            assert binary.limite_superior(x)[0]==bisect.bisect_right(serie.instantes,x)

    class Espia(MetodoBusca):
        def __init__(self):
            super().__init__(); self.chamadas=[]; self.preparacoes=0
        def _preparar(self): self.preparacoes+=1
        def _buscar(self,x):
            self.chamadas.append(x)
            return 0 if len(self.serie)>0 else -1
    svc=ServicoConsultas(); espia=Espia()
    svc.registrar_sensor(s,espia)
    assert svc.consultar('S1',2).status=='encontrada'
    assert espia.chamadas==[2], 'O serviço não delegou à estratégia'
    svc.consultar('S1',2)
    assert espia.preparacoes==1, 'Não preparar em cada consulta'
    esperar_erro(ValueError,lambda:svc.registrar_sensor(serie_de([99],'S2'),espia))
    # Substituir uma estratégia não deve exigir alterações no serviço.
    for _,fabrica in fabricas(3).items():
        svc.registrar_sensor(s,fabrica())
        assert svc.consultar('S1',4).medicao==s.medicoes[1]
    s2=serie_de([100,200,300],'S2')
    svc.registrar_sensor(s,BuscaIndexada(2)); svc.registrar_sensor(s2,BuscaIndexada(1))
    lote=[('S1',4),('S2',200),('S1',3),('XX',4),('S2',100)]
    respostas=svc.consultar_lote(lote)
    assert [r.status for r in respostas]==['encontrada','encontrada','instante_ausente','sensor_inexistente','encontrada']
    assert respostas[0].medicao.sensor_id=='S1' and respostas[1].medicao.sensor_id=='S2'
    assert respostas[2].resultado is not None and respostas[3].resultado is None
    for inicio,fim in [(-100,999),(2,2),(3,5),(7,10),(0,1)]:
        r=svc.investigar_periodo('S1',inicio,fim)
        ref=referencia_periodo(s,inicio,fim)
        assert isinstance(r,ResumoPeriodo) and r.medicoes==ref
        assert r.quantidade==len(ref)
        assert r.minimo==min((m.valor for m in ref),default=None)
        assert r.maximo==max((m.valor for m in ref),default=None)
        assert r.media==(sum(m.valor for m in ref)/len(ref) if ref else None)
    vazio=serie_de([],'V'); svc.registrar_sensor(vazio,BuscaBinaria())
    assert svc.investigar_periodo('V',0,10).quantidade==0
    assert svc.consultar('V',0).status=='instante_ausente'
    esperar_erro(KeyError,lambda:svc.investigar_periodo('XX',0,10))
    esperar_erro(ValueError,lambda:svc.investigar_periodo('S1',10,0))
    print(f'{consultas_verificadas} consultas pontuais verificadas; testes OO, limites e serviço aprovados.')
if implementado(): testar()


### Acrescente dois testes seus
  Em uma célula Python, acrescente casos que não sejam simples cópias dos testes fornecidos. Sugestões: instantes inteiros muito grandes; substituição do histórico de um sensor; janela entre duas amostras; reuso de estratégia em série com outro tamanho. Descreva o erro que cada teste pretende detectar.


**Justificativa dos testes próprios**

_Escreva sua resposta aqui._


## 11. Experimentos prontos — execute e interprete
  Cada cenário usa as mesmas consultas para todos os métodos. Metade é presente e metade ausente; a semente é fixa. Há três tamanhos (100, 1.000, 5.000), mantendo o tempo de execução acessível. Comece pelos testes.

  A preparação das estratégias é medida separadamente; a criação/validação de SerieSensor e a construção das instâncias ficam fora desse tempo. Em cada cenário a criação da série é comum a todos. A consulta é medida em lotes, depois de aquecimento, com três repetições: mostramos mediana e mínimo. Geração, validação e gráficos ficam fora do trecho cronometrado. As consultas medidas **incluem a instrumentação**, a chamada de método e a criação de ResultadoBusca.

  `entradas_indice` mede quantidade de entradas, **não bytes nem memória total**. O vetor comum de registros e instantes ocupa O(n); o índice é memória adicional. Não confunda tempo observado com uma prova de complexidade.


In [ ]:
def gerar_serie(n,cenario,sensor_id='TEMP_01'):
    rng=np.random.default_rng(503+n)
    if cenario=='periodico': tempos=np.arange(n,dtype=np.int64)*10
    elif cenario=='falhas':
        k=int(.9*n)
        tempos=np.concatenate([np.arange(k,dtype=np.int64)*10,
                               10**8+np.arange(n-k,dtype=np.int64)*10000])
    elif cenario=='eventos':
        # Passos curtos predominam; rajadas são separadas por pausas longas.
        passos=rng.choice([10,20,10000],size=n,p=[.80,.18,.02])
        tempos=np.cumsum(passos)
    else: raise ValueError(cenario)
    valores=60+3*np.sin(np.arange(n)/30)+rng.normal(0,.3,n)
    return SerieSensor(sensor_id,[Medicao(sensor_id,int(t),float(v)) for t,v in zip(tempos,valores)])

def gerar_consultas(serie,perfil,seed,quantidade=80):
    rng=np.random.default_rng(seed)
    limite=max(1,len(serie)//100) if perfil=='inicio' else len(serie)
    ids=rng.integers(0,limite,size=quantidade//2)
    presentes=[serie.instantes[int(i)] for i in ids]
    # Instantes gerados são múltiplos de dez, então +1 garante ausência.
    registros=[(x,'presente') for x in presentes]+[(x+1,'ausente') for x in presentes]
    rng.shuffle(registros)
    return [x for x,t in registros],[t for x,t in registros]

def comparar_metodos(serie,qs,tipos,bloco=None,nomes=None,repeticoes=3):
    b=max(1,isqrt(len(serie))) if bloco is None else bloco
    registros=[]; tempos=[]
    for nome,fabrica in fabricas(b).items():
        if nomes and nome not in nomes: continue
        preparacoes=[]
        for _ in range(repeticoes):
            metodo=fabrica()
            t0=perf_counter(); metodo.preparar(serie); preparacoes.append(perf_counter()-t0)
        esperado=[referencia_pontual(serie,x) for x in qs]
        respostas=[metodo.buscar(x) for x in qs]
        assert [r.posicao for r in respostas]==esperado, nome
        for j,r in enumerate(respostas):
            registros.append(dict(metodo=nome,tipo=tipos[j],comparacoes=r.comparacoes,n=len(serie),bloco=b))
        # Aquecimento separado; nenhuma validação, preparação ou geração dentro do trecho cronometrado.
        for x in qs: metodo.buscar(x)
        duracoes=[]
        for _ in range(repeticoes):
            t0=perf_counter()
            for x in qs: metodo.buscar(x)
            duracoes.append(perf_counter()-t0)
        tempos.append(dict(metodo=nome,n=len(serie),bloco=b,
                           preparacao_us=median(preparacoes)*1e6,
                           lote_us=median(duracoes)*1e6,
                           consulta_us=median(duracoes)*1e6/len(qs),
                           lote_min_us=min(duracoes)*1e6,entradas_indice=metodo.entradas_indice))
    return pd.DataFrame(registros),pd.DataFrame(tempos)

print('Geradores e medidores prontos. O custo de construir SerieSensor fica fora da preparação das estratégias.')


### Investigação 1 — Periodicidade, falhas e eventos
  Compare médias por presença/ausência e distribuição. Responda: (1) quando interpolação é favorável? (2) onde perde vantagem? (3) há divergência entre comparação e tempo? (4) por que um cenário sintético não sustenta superioridade universal?


In [ ]:
if implementado():
    resultados=[]; tempos=[]
    for n in [100,1000,5000]:
        for cenario in ['periodico','falhas','eventos']:
            s=gerar_serie(n,cenario)
            qs,tipos=gerar_consultas(s,'espalhado',503+n)
            r,t=comparar_metodos(s,qs,tipos)
            r['cenario']=cenario; t['cenario']=cenario
            resultados.append(r); tempos.append(t)
    e1=pd.concat(resultados,ignore_index=True); t1=pd.concat(tempos,ignore_index=True)
    tabela=e1.groupby(['cenario','n','metodo','tipo']).comparacoes.mean().unstack()
    display(tabela.round(2))
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,cenario in zip(axes,['periodico','falhas','eventos']):
        resumo=e1[e1.cenario==cenario].groupby(['n','metodo']).comparacoes.mean().reset_index()
        for nome,g in resumo.groupby('metodo'): ax.plot(g.n,g.comparacoes,'o-',label=nome)
        ax.set(xscale='log',yscale='log',xlabel='Medições por sensor',ylabel='Comparações médias',title=cenario)
        ax.grid(alpha=.2)
    axes[-1].legend(fontsize=8); fig.suptitle('Distribuição dos instantes'); fig.tight_layout(); plt.show()
    display(t1[t1.n==5000].round(2))


**Análise 1**

_Escreva sua resposta aqui._


### Investigação 2 — Índice: preparação, blocos e amortização
  Observe a consulta e a quantidade de entradas para vários blocos. Compare a indexada com salto e binária. Para uma estimativa simples, use `Ttotal(Q)=Tpreparacao+Q*Tconsulta`.

  Se a indexada tem menor tempo por consulta que uma alternativa, uma preparação adicional pode ser amortizada. Se não tiver, **não existe ponto de amortização observado neste modelo**. Não force uma resposta positiva.

  Responda: (1) que região de b equilibra as comparações? (2) qual custo de memória não aparece no gráfico? (3) a construção compensa frente à binária nestes resultados? (4) como mudanças no histórico afetam o índice?


In [ ]:
if implementado():
    s=gerar_serie(5000,'periodico'); qs,tipos=gerar_consultas(s,'espalhado',600)
    contagens=[]; tempos=[]
    for b in [1,5,10,25,50,70,100,250,500,5000]:
        r,t=comparar_metodos(s,qs,tipos,b,['indexada','salto'])
        contagens.append(r); tempos.append(t)
    e2=pd.concat(contagens,ignore_index=True); t2=pd.concat(tempos,ignore_index=True)
    display(t2.round(2))
    resumo=e2.groupby(['metodo','bloco']).comparacoes.mean().reset_index()
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for nome,g in resumo.groupby('metodo'): axes[0].plot(g.bloco,g.comparacoes,'o-',label=nome)
    axes[0].axvline(isqrt(len(s)),color='gray',linestyle='--',label='sqrt(n)')
    axes[0].set(xscale='log',yscale='log',xlabel='Bloco',ylabel='Comparações médias'); axes[0].legend()
    for nome,g in t2.groupby('metodo'): axes[1].plot(g.bloco,g.entradas_indice,'o-',label=nome)
    axes[1].set(xscale='log',xlabel='Bloco',ylabel='Entradas de índice'); axes[1].legend()
    fig.suptitle('Blocos: custo de consulta e memória adicional'); fig.tight_layout(); plt.show()
    base=t1[(t1.n==5000)&(t1.cenario=='periodico')].set_index('metodo')
    preparacao=base.loc['indexada','preparacao_us']-base.loc['binaria','preparacao_us']
    economia=base.loc['binaria','consulta_us']-base.loc['indexada','consulta_us']
    if economia>0:
        estimativa=max(0,preparacao)/economia
        print('Estimativa de consultas para amortizar frente à binária:',round(estimativa,2))
    else:
        print('Sem amortização observada frente à binária: a indexada não teve menor tempo por consulta neste cenário.')


**Análise 2**

_Escreva sua resposta aqui._


### Investigação 3 — Posição das consultas
  Compare exponencial e binária com consultas no primeiro 1% e espalhadas. Responda: (1) em que cenários a exponencial aproveita a posição? (2) como isso se relaciona à posição de inserção de chave ausente? (3) primeiro 1% implica custo constante quando n cresce? (4) como a composição com a binária afeta a interpretação de tempo em Python?


In [ ]:
if implementado():
    partes=[]; tempos=[]
    for n in [100,1000,5000]:
        s=gerar_serie(n,'periodico')
        for perfil in ['inicio','espalhado']:
            qs,tipos=gerar_consultas(s,perfil,900+n)
            r,t=comparar_metodos(s,qs,tipos,nomes=['binaria','exponencial'])
            r['perfil']=perfil; t['perfil']=perfil; partes.append(r); tempos.append(t)
    e3=pd.concat(partes,ignore_index=True); t3=pd.concat(tempos,ignore_index=True)
    display(e3.groupby(['perfil','n','metodo','tipo']).comparacoes.mean().unstack().round(2))
    display(t3.round(2))
    fig,ax=plt.subplots(figsize=(8,4))
    e3[e3.n==5000].groupby(['perfil','metodo']).comparacoes.mean().unstack().plot.bar(ax=ax,rot=0)
    ax.set(ylabel='Comparações médias',title='Consultas no primeiro 1% versus espalhadas')
    fig.tight_layout(); plt.show()


**Análise 3**

_Escreva sua resposta aqui._


## 12. Janela de investigação industrial
  Execute a demonstração, que chama seu serviço, e valide o resumo. Explique: (1) por que a consulta pontual pode usar interpolação enquanto a janela usa limites binários? (2) por que retornar todos os registros tem custo dependente de k? (3) por que mínimo, máximo e média não indicam sozinhos a causa de uma ocorrência?


In [ ]:
if implementado():
    serie=gerar_serie(5000,'eventos','VIB_01')
    servico=ServicoConsultas(); servico.registrar_sensor(serie,BuscaInterpolacao())
    inicio=serie.instantes[1500]; fim=serie.instantes[1600]
    resumo=servico.investigar_periodo('VIB_01',inicio,fim)
    assert resumo.medicoes==referencia_periodo(serie,inicio,fim)
    print('Resumo do período:',resumo.quantidade,'medições;',resumo.comparacoes,'comparações nos limites')
    print('Mínimo:',round(resumo.minimo,3),'Máximo:',round(resumo.maximo,3),'Média:',round(resumo.media,3))
    fig,ax=plt.subplots(figsize=(10,3))
    ax.plot([m.instante for m in resumo.medicoes],[m.valor for m in resumo.medicoes])
    ax.set(xlabel='Instante (unidade arbitrária)',ylabel='Valor sintético',title='Janela da investigação industrial')
    fig.tight_layout(); plt.show()


**Análise da janela**

_Escreva sua resposta aqui._


## 13. Recomendação técnica e entrega
  Escreva **400–600 palavras** recomendando estratégias para os três cenários e para os perfis de consulta. Inclua ao menos três evidências numéricas extraídas das suas tabelas; explique a arquitetura OO, preparação, memória, tempo, limites das medições e uma escolha que você alteraria se os dados fossem atualizados frequentemente.

  Antes de entregar: preencher identificação; reiniciar a sessão e executar tudo em ordem; confirmar testes; manter gráficos e respostas; fazer download do `.ipynb`. Registre uso de ferramentas externas conforme a política da disciplina. Um notebook com apenas saídas, sem suas implementações e justificativas, não constitui a entrega.


**Recomendação técnica**

_Escreva sua resposta aqui._


## Referências
  - Cormen et al. **Introduction to Algorithms**, 4ª ed., MIT Press, 2022: análise e invariantes.
  - Knuth. **The Art of Computer Programming, Volume 3: Sorting and Searching**, 2ª ed., 1998.
  - Perl, Itai e Avni. **Interpolation Search—A Log LogN Search**, CACM 21(7), 1978. [DOI](https://doi.org/10.1145/359545.359557).
  - Bentley e Yao. **An Almost Optimal Algorithm for Unbounded Searching**, IPL 5(3), 1976. [DOI](https://doi.org/10.1016/0020-0190(76)90071-5).
  - Python: [classes abstratas](https://docs.python.org/3/library/abc.html), [dataclasses](https://docs.python.org/3/library/dataclasses.html), [bisect](https://docs.python.org/3/library/bisect.html), [timeit e cuidados de medição](https://docs.python.org/3/library/timeit.html).

  Pseudocódigos adaptados ao contrato didático desta atividade. A infraestrutura usa `perf_counter` e apresenta mínimo e mediana das repetições; execute novamente se as diferenças forem pequenas ou inconsistentes.
